# Web Classifier
Phan loai web: SSR / CSR / Anti-bot

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Crawl_Data/CrawlData'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Work:', WORK_DIR)

Mounted at /content/drive
Work: /content/drive/MyDrive/Crawl_Data/CrawlData


In [3]:
!pip install requests beautifulsoup4 lxml -q

In [4]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime

INPUT_FILE = os.path.join(WORK_DIR, 'input_urls.json')
SSR_OUTPUT = os.path.join(WORK_DIR, 'ssr_urls.json')
CSR_OUTPUT = os.path.join(WORK_DIR, 'csr_urls.json')
ANTIBOT_OUTPUT = os.path.join(WORK_DIR, 'antibot_urls.json')
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

In [5]:
def classify_url(url, timeout=15, selector=None):
    result = {'url': url, 'web_type': 'ssr', 'reason': None}
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        html = r.text.lower()
        soup = BeautifulSoup(r.text, 'lxml')

        # 1. Anti-bot Check (Hard blocks)
        antibot_signs = ['cloudflare', 'captcha', 'recaptcha', 'hcaptcha', 'challenge-platform', 'please wait', 'checking your browser', 'ddos-guard', 'sucuri', '__cf_bm', 'just a moment']
        if r.status_code in [403, 429, 503] or any(sign in html for sign in antibot_signs):
            result['web_type'] = 'antibot'
            result['reason'] = f'Anti-bot detected (status: {r.status_code})'
            return result

        # 2. Selector Check (The most reliable generic method)
        # If user provided a selector, we test if it exists in the raw HTML.
        if selector and selector.get('selector') and selector.get('type') == 'css':
            try:
                matches = soup.select(selector['selector'])
                if len(matches) > 0:
                    result['web_type'] = 'ssr'
                    result['reason'] = f'Selector matched {len(matches)} elements in raw HTML'
                    return result
                else:
                    # Selector provided but NOT found in raw HTML -> Strong signal for CSR
                    # But we verify it's not just a bad selector by checking for generic JS signs
                    result['web_type'] = 'csr'
                    result['reason'] = 'Selector NOT found in raw HTML (needs JS?)'
                    # We don't return immediately, we can double check generic signs below if we want,
                    # but usually this is enough to justify using a browser.
                    # Exception: simple script-based content loading (document.write, etc)
                    return result
            except: pass

        # 3. Generic CSR Heuristics (if no selector provided or generic check needed)

        # A. Framework Signatures
        csr_signs = ['__next_data__', 'data-reactroot', 'data-reactid', '__nuxt__', '__vue__', 'ng-app', 'ng-controller', 'window.__initial_state__', 'window.__preloaded_state__']
        if any(sign in html for sign in csr_signs):
            result['web_type'] = 'csr'
            result['reason'] = 'JS Framework Signature'
            return result

        # B. Script Redirection / Cookie Setting (Generic pattern for simple bot protection/CSR)
        if 'document.cookie' in html and ('window.location.reload' in html or 'location.href' in html):
            result['web_type'] = 'csr'
            result['reason'] = 'Script-based Redirect/Cookie'
            return result

        # C. Content vs Script Ratio
        scripts = soup.find_all('script')
        script_size = sum(len(s.get_text()) for s in scripts if s.get_text())
        text_content = soup.get_text(strip=True)

        if len(text_content) < 500 and script_size > 1000:
            result['web_type'] = 'csr'
            result['reason'] = 'Low content / High script ratio'
            return result

        # Default to SSR if it looks like a normal page and we passed Anti-bot checks
        result['reason'] = 'Standard HTML Structure'

    except requests.exceptions.Timeout:
        result['web_type'] = 'antibot'
        result['reason'] = 'Timeout'
    except Exception as e:
        result['web_type'] = 'ssr'
        result['reason'] = f'Error: {str(e)[:30]}'

    return result

In [6]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

urls = data.get('urls', [])
options = data.get('options', {})
results = {'ssr': [], 'csr': [], 'antibot': []}

print(f'URLs: {len(urls)}')
print('='*50)

for i, cfg in enumerate(urls, 1):
    url = cfg.get('url')
    wtype = cfg.get('web_type', 'auto')
    selector = cfg.get('level2_selector') or cfg.get('region_selector')

    # Generic: Try to find ANY selector in the config to test
    if not selector and cfg.get('levels'):
        for lvl in cfg['levels']:
            if lvl.get('selector'):
                selector = lvl['selector']
                break

    print(f'[{i}] {url[:50]}...', end=' ')

    if wtype in ['ssr', 'csr', 'antibot']:
        c = {'url': url, 'web_type': wtype, 'reason': 'Manual'}
    else:
        c = classify_url(url, options.get('timeout', 30), selector)

    print(f'-> {c["web_type"].upper()} ({c["reason"]})')

    full_cfg = cfg.copy()
    full_cfg['web_type'] = c['web_type']
    full_cfg['classification_reason'] = c['reason']

    if c['web_type'] in results:
        results[c['web_type']].append(full_cfg)

print('='*50)
print(f'SSR: {len(results["ssr"])} | CSR: {len(results["csr"])} | Antibot: {len(results["antibot"])}')

URLs: 1
[1] https://bqp.vn/home/vbpl... -> CSR (Selector NOT found in raw HTML (needs JS?))
SSR: 0 | CSR: 1 | Antibot: 0


In [7]:
def save(urls_list, path):
    output_data = {'urls': urls_list, 'options': options, 'classified_at': datetime.now().isoformat()}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)
    print(f'Saved: {path} ({len(urls_list)} URLs)')

if results['ssr']: save(results['ssr'], SSR_OUTPUT)
if results['csr']: save(results['csr'], CSR_OUTPUT)
if results['antibot']: save(results['antibot'], ANTIBOT_OUTPUT)

print('\nDone! Run crawler:')
if results['ssr']: print('  - 2_ssr_crawler.ipynb')
if results['csr']: print('  - 3_csr_crawler.ipynb')
if results['antibot']: print('  - 4_antibot_crawler.ipynb')

Saved: /content/drive/MyDrive/Crawl_Data/CrawlData/csr_urls.json (1 URLs)

Done! Run crawler:
  - 3_csr_crawler.ipynb
